# Clean, Source-Organized Copy
Saved outputs and machine-specific metadata are intentionally excluded. The byte-identical original remains in the local audit archive. Prepare the local workbench before running this notebook. Notebooks 01 and 02 execute GPU inference when explicitly enabled; 03 disables its optional GPU activation check by default.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "scripts" / "lab.py").is_file()
)
WORKSPACE = PROJECT_ROOT / ".local" / "workbench"
if not (WORKSPACE / "weight_edit.py").is_file():
    raise RuntimeError("Run python scripts/lab.py prepare --with-local-data first.")
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))


# Controlled experiments and a stopping rule

Compare the same development questions and per-question seeds across six predeclared conditions. **A final answer, natural EOS stop, and correct arithmetic are separate measurements.** A correct boxed number inside unfinished thinking is not a delivered answer.

By default this notebook only reads the latest local pilot. Set `RUN_PILOT = True` to deliberately start another GPU run. No new packages, model downloads, training, commits, or uploads occur.

In [ ]:
import json
from pathlib import Path
from statistics import mean

import torch

from experiment_design import CONDITIONS, DEVELOPMENT
from experiment_runner import load_local_model, run_suite
from math_probe import PROTOCOL_VERSION

RUN_PILOT = False
PROBE_COUNT = 2
MAX_NEW_TOKENS = 1024
WALL_BUDGET_SECONDS = 900
TOTAL_TOKEN_BUDGET = 12288
REPORT_PATH = None

## Predeclared experiment

- Original model with thinking on; original model with thinking off.
- Exact published edit; layer 15 only; layer 19 only; half-strength edit.
- Same questions, per-question seeds, presence penalty, temperature, and token cap. The thinking flag changes only in the declared thinking-off control.
- Half-strength interpolation changes only the published sparse indices, with BF16 rounding. Its hash is recorded as a new variant, not falsely labeled as the published checkpoint.
- Each edit is rolled back and hash-verified. Every generation is journaled before starting the next one. Token/time limits stop scheduling work; an interrupted or incomplete pilot cannot select a candidate.

The 20 handmade held-out math questions and 10 basic instruction/QA controls are separate from development. They are not benchmark datasets and have not been used to select a variant.

In [ ]:
for condition in CONDITIONS:
    print(condition.name, '| thinking:', condition.thinking,
          '| layers:', condition.layers, '| strength:', condition.strength)
print('\nProtocol:', PROTOCOL_VERSION)
print('Soft wall-time limit:', WALL_BUDGET_SECONDS, 'seconds')
print('Maximum generated-token allocation:', TOTAL_TOKEN_BUDGET)

In [ ]:
if RUN_PILOT:
    if type(PROBE_COUNT) is not int or not 1 <= PROBE_COUNT <= len(DEVELOPMENT):
        raise ValueError('Choose between 1 and 4 development problems.')
    model, tokenizer = load_local_model()
    try:
        run_dir, report = run_suite(
            model, tokenizer, problems=DEVELOPMENT[:PROBE_COUNT],
            max_new_tokens=MAX_NEW_TOKENS, seconds=WALL_BUDGET_SECONDS,
            token_budget=TOTAL_TOKEN_BUDGET,
        )
        REPORT_PATH = run_dir / 'summary.json'
    finally:
        del model
        torch.cuda.empty_cache()
else:
    print('GPU pilot skipped. Reading an existing local result only.')

In [ ]:
results_root = (Path.cwd() / 'results').resolve()
if REPORT_PATH is None:
    candidates = list(results_root.glob('pilot_*/summary.json'))
    if not candidates:
        raise FileNotFoundError('No finished pilot report yet. Wait for the running pilot, or enable RUN_PILOT.')
    REPORT_PATH = max(candidates, key=lambda path: path.stat().st_mtime)
REPORT_PATH = Path(REPORT_PATH).resolve()
if not REPORT_PATH.is_relative_to(results_root) or REPORT_PATH.name != 'summary.json':
    raise ValueError('Select a summary.json inside this project results directory.')
report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
if report.get('protocol') != PROTOCOL_VERSION or report.get('stage') != 'pilot':
    raise ValueError('This is not a pilot report for the current protocol.')
print('Report:', REPORT_PATH.relative_to(Path.cwd()))
print('Status:', report['status'], '| completed:', report['completed_generations'], '/', report['expected_generations'])
print('Original weights restored:', report['restored_base_hashes_verified'])

In [ ]:
print(f"{'condition':<19} {'delivered':<12} {'EOS':<8} {'cap':<8} {'tokens':<9} {'repeat4':<9} {'sec':<8}")
for condition in CONDITIONS:
    row = report['summary'].get(condition.name)
    if row is None:
        continue
    records = [item for item in report['records'] if item['condition'] == condition.name]
    repeat = mean(item['repeated_4gram_fraction'] for item in records)
    print(f"{condition.name:<19} {str(row['delivered_correct']) + '/' + str(row['count']):<12} "
          f"{row['natural_stop_rate']:<8.0%} {row['cap_rate']:<8.0%} "
          f"{row['mean_tokens']:<9.1f} {repeat:<9.1%} {row['mean_seconds']:<8.1f}")
print('\nrepeat4 is only a repetition heuristic. Times are local observations, not speed benchmarks.')

## Stop or proceed?

A complete, paired development pilot must show at least a **25-percentage-point delivered-correct gain over the better of thinking-on and thinking-off**, without reducing natural completion versus thinking-on. This is a compute-allocation gate, not a statistical significance claim. With two questions, estimates are very coarse.

If the gate fails, stop this candidate search and leave the held-out set unused. If it passes, freeze the selected edit and token cap before a separate held-out run with two seed groups. Do not expand the search until a favorable example appears.

In [ ]:
decision = report['decision']
print(decision['reason'])
if decision['allow_holdout']:
    print('Frozen candidate:', decision['selected_condition'])
    print('A held-out run is separate and may take hours. Inspect its budget before starting.')
    print('Command: .venv\\Scripts\\python.exe experiment_runner.py --stage holdout --selection',
          REPORT_PATH.relative_to(Path.cwd()))
else:
    print('STOP: no held-out inference is authorized by this pilot decision.')

In [ ]:
for name in ('base_thinking', 'base_no_thinking', 'published_both'):
    sample = next((row for row in report['records'] if row['condition'] == name), None)
    if sample:
        print('\n---', name, '/', sample['problem_id'], '---')
        print('Stop:', sample['stop_reason'], '| final correct:', sample['correct'],
              '| boxed reasoning correct:', sample['reasoning_boxed_correct'])
        print(sample['text'][:1500])